Esercizio 1 – Dataset clienti
•	Colonne: Nome, email, data_nascita, età, stipendio, città
•	Pulisci stringhe e email, valida pattern email
•	Converti le date in datetime e calcola l’età reale
•	Imputa valori mancanti con media/mediana
•	Identifica outlier nelle colonne numeriche
•	Trasforma colonne ripetute in tipo categorico

In [10]:
from datetime import datetime, timedelta
import random 
import pandas as pd
import numpy as np

###########################################################
# CREAZIONE DATI CASUALI PER IL DATASET
###########################################################

# 1. Imposta il numero di record per il dataset 
n_righe = 10

# 2. Liste di supporto per i dati casuali
nomi = ["Marco", "Giulia", "Luca", "Francesca", "Alessandro", "Sofia", "Andrea", "Chiara", "Matteo", "Elena"]
cognomi = ["Rossi", "Bianchi", "Russo", "Ferrari", "Esposito", "Colombo", "Ricci", "Marino", "Bruno", "Conti"]
citta_list = ["Milano", "Roma", "Torino", "Napoli", "Bologna", "Firenze", "Palermo", "Verona", "Genova", "Bari"]
domini_email = ["gmail.com", "yahoo.it", "outlook.com", "hotmail.it"]

dati = []
oggi = datetime.now()

# 3. Generazione casuale dei record 
for _ in range(n_righe):
    # Nome e cognome 
    nome_completo = f"{random.choice(nomi)} {random.choice(cognomi)}"

    # Età casuale tra 18 e 65 anni 
    eta = random.randint(18, 65)

    # Data di nascita coerente con l'età generata
    giorni_offset = random.randint(0, 364)
    data_nascita = (oggi - timedelta(days = eta * 365 + giorni_offset)).date()

    # Email formattata nome.congome@dominio
    email = f"{nome_completo.lower().replace(' ','.')}@{random.choice(domini_email)}"

    # Stipendio casuale arrotondato 
    stipendio = round(random.uniform(18000.0, 70000.0), 2)

    # Aggiunta del record 
    dati.append({
        "Nome": nome_completo,
        "email": email,
        "data_nascita": data_nascita.strftime("%Y-%m-%d"),
        "età":eta,
        "stipendio":stipendio,
        "città":random.choice(citta_list)
    })

# 4. creazione dataframe 

df = pd.DataFrame(dati)

print(df)

###########################################################
# PULIZIA STRINGHE E VALIDAZIONE MAIL
###########################################################

# capitalize() rende maiuscola solo la prima lettera della stringa
df["Nome"] = df["Nome"].map(lambda x: x.strip().title() if pd.notnull(x) else x)

df["email"] = df["email"].map(lambda x: x.strip().lower() if pd.notnull(x) else x)

df["email_valida"] = df["email"].str.contains(r"^\w+[\.-]?\w+@\w+\.\w+$", na= False)

df["stipendio"] = df["stipendio"].fillna(df["stipendio"].median())

df["età"] = df["età"].fillna(df["età"].median())

###########################################################
# IDENTIFICAZIONE OUTLIER NELLE COLONNE NUMERICHE
###########################################################

# Metodo IQR 

Q1 = df["stipendio"].quantile(0.25)
Q3 = df["stipendio"].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR 
lim_sup = Q3 + 1.5 * IQR

outliers_stipendio = df[(df["stipendio"] < lim_inf) | (df["stipendio"] > lim_sup)]

Q1 = df["età"].quantile(0.25)
Q3 = df["età"].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR 
lim_sup = Q3 + 1.5 * IQR

outliers_eta = df[(df["età"] < lim_inf) | (df["età"] > lim_sup)]

print(f"OUTLIER STIPENDIO: {outliers_stipendio}")
print(f"OUTLIER ETà: {outliers_eta}")

###########################################################
# IMPOSTAZIONE CATEGORY
###########################################################

df["città"] = df["città"].astype("category")


                 Nome                         email data_nascita  età  \
0         Luca Marino       luca.marino@outlook.com   2006-05-17   20   
1    Alessandro Rossi   alessandro.rossi@hotmail.it   1990-06-12   36   
2       Chiara Marino     chiara.marino@outlook.com   1976-04-12   50   
3      Giulia Bianchi       giulia.bianchi@yahoo.it   1968-03-07   58   
4  Francesca Esposito  francesca.esposito@gmail.com   1988-06-25   38   
5      Marco Esposito      marco.esposito@gmail.com   1963-11-19   62   
6       Elena Ferrari     elena.ferrari@outlook.com   1972-12-24   53   
7   Alessandro Marino   alessandro.marino@gmail.com   2004-06-14   22   
8         Luca Marino       luca.marino@outlook.com   2001-02-12   25   
9     Chiara Esposito    chiara.esposito@hotmail.it   2002-10-27   23   

   stipendio    città  
0   22306.87   Milano  
1   48874.34   Torino  
2   61314.22   Napoli  
3   48404.17   Milano  
4   55369.99     Bari  
5   28593.84   Milano  
6   63287.27  Firenze  
7   

Esercizio 2 – Dataset vendite
•	Colonne: prodotto, categoria, prezzo, quantità, data_vendita
•	Pulizia stringhe e uniformazione categorie
•	Conversione data e calcolo giorni dalla prima vendita
•	Rilevamento outlier su prezzo e quantità 
•	Creazione di feature: ricavo totale, log del prezzo, interazioni tra quantità e mese
•	Ottimizzazione memoria convertendo colonne appropriate in category e numeriche più piccole

In [23]:
import pandas as pd 
import numpy as np
from datetime import datetime, timedelta


###########################################################
# CREAZIONE DATI CASUALI PER IL DATASET
###########################################################

# impostazione del seed per la riproducibilità

np.random.seed(42)

n_righe = 10

# Categorie e prodotti correalti

prodotti_per_categoria = {
    'Elettronica': ['Smartphone', 'Laptop', 'Cuffie Bluetooth', 'Smartwatch', 'Tablet'],
    'Abbigliamento': ['Maglietta', 'Giacca', 'Pantaloni', 'Scarpe Sportive', 'Cappello'],
    'Casa e Cucina': ['Caffettiera', 'Frullatore', 'Set Pentole', 'Lampada da Tavolo', 'Aspirapolvere'],
    'Libri': ['Romanzo Thriller', 'Manuale Python', 'Libro di Cucina', 'Saggio Storico', 'Fumetto'],
    'Sport': ['Pallone da Calcio', 'Tappetino Yoga', 'Manubri', 'Bicicletta', 'Borraccia Termica']
}

prezzi_range = {
    'Elettronica': (50.0, 1200.0),
    'Abbigliamento': (15.0, 150.0),
    'Casa e Cucina': (20.0, 250.0),
    'Libri': (10.0, 45.0),
    'Sport': (8.0, 300.0)
}

categorie = list(prodotti_per_categoria.keys())
dati = [] 
data_inizio = datetime(2025, 1, 1)

for _ in range(n_righe):
    categ = np.random.choice(categorie)
    prod = np.random.choice(prodotti_per_categoria[categ])
    p_min, p_max = prezzi_range[categ]
    prezzo = round(np.random.uniform(p_min, p_max), 2)
    quantita = np.random.randint(1, 11)

    giorni_casuali = np.random.randint (0, 365)
    data_vendita = (data_inizio + timedelta(days = int(giorni_casuali))).strftime('%Y-%m-%d') # data stringa formattata

    # Dataset
    dati.append(
        {
            'prodotto': prod,
            'categoria': categ,
            'prezzo': prezzo,
            'quantità': quantita,
            'data_vendita': data_vendita
        }
    )

df = pd.DataFrame(dati) # creazione DataFrame

# inserimento casuale di qualche valore nullo 
df.loc[np.random.choice(df.index, size = 5, replace=False), 'prezzo'] = np.nan
df.loc[np.random.choice(df.index, size = 3, replace=False), 'quantità'] = np.nan

print("\n######## DATAFRAME ORIGINALE ##########")
print(df)

###########################################################
# PULIZIA STRINGHE E UNIFORMITA' CATEGORIE 
###########################################################

df["prodotto"] = df["prodotto"].map(lambda x : x.strip().title() if pd.notnull(x) else x)
df["categoria"] = df["categoria"].map(lambda x : x.strip().title() if pd.notnull(x) else x)
df["prodotto"] = df["prodotto"].astype("category")
df["categoria"] = df["categoria"].astype("category")

###########################################################
# CONVERSIONE DATA E CALCOLO GIORNI DALLA PRIMA VENDITA
###########################################################

df["data_vendita"] = pd.to_datetime(df["data_vendita"])

# Giorni dalla prima vendita per ciascun prodotto 

oggi = pd.Timestamp.now().normalize()

df["giorni_prima_vendita_prodotto"] = (
    oggi - df.groupby("prodotto")["data_vendita"].transform("min")
).dt.days

# Gestisco i NAN per prezzo e quantità 

df["prezzo"] = df["prezzo"].fillna(
    df.groupby('categoria')['prezzo'].transform('median')
)

df["quantità"] = df["quantità"].fillna(df["quantità"].median())

print("\n################ DATAFRAME DOPO GESTIONE NAN ###################")
print(df)

###########################################################
# RILEVAMENTO OUTLIER SU PREZZO E QUANTITA'
###########################################################

Q1 = df["prezzo"].quantile(0.25)
Q3 = df["prezzo"].quantile(0.75)
IQR = Q3 - Q1 

limite_inf = Q1 - 1.5 * IQR 
limite_sup = Q3 + 1.5 * IQR 

df["is_outlier_prezzo"] = (df["prezzo"] < limite_inf) | (df["prezzo"] > limite_sup)

Q1 = df["quantità"].quantile(0.25)
Q3 = df["quantità"].quantile(0.75)
IQR = Q3 - Q1 

limite_inf = Q1 - 1.5 * IQR 
limite_sup = Q3 + 1.5 * IQR 

df["is_outlier_quantità"] = (df["quantità"] < limite_inf) | (df["quantità"] > limite_sup)

print("\n################ DATAFRAME DOPO GESTIONE NAN E OULIERS ###################")
print(df)

###########################################################
# CREAZIONE NUOVE FEATURE
# ricavo totale, log del prezzo, interazioni tra quantità e mese
###########################################################

# Ricavo totale 
df["ricavo_totale"] = df["quantità"] * df["prezzo"]

# Applicazione del logaritmo (log(1 + x)) per gestire i prezzi
df["log_prezzo"] = np.log1p(df["prezzo"]) 

# Controllo del risultato
print("\n################ DATAFRAME CON LOG PREZZO ###################")
print(df[["prodotto", "categoria", "prezzo", "log_prezzo"]].head())

# Interazione tra Quantità e Mese
df["anno_mese"] = df["data_vendita"].dt.to_period("M")
# la somma dei ricavi per ogni combinazione di Categoria, Prodotto e Mese.
mensile = df.groupby(["categoria","prodotto","anno_mese"])["ricavo_totale"].sum().reset_index()

print("\n#######  la somma dei ricavi per ogni combinazione di Categoria, Prodotto e Mese ########")
print(mensile)



######## DATAFRAME ORIGINALE ##########
            prodotto      categoria  prezzo  quantità data_vendita
0            Fumetto          Libri     NaN       8.0   2025-07-08
1     Tappetino Yoga          Sport   53.55       8.0   2025-04-10
2      Aspirapolvere  Casa e Cucina     NaN       NaN   2025-07-11
3            Fumetto          Libri   31.61      10.0   2025-01-22
4         Bicicletta          Sport     NaN       NaN   2025-07-07
5  Lampada da Tavolo  Casa e Cucina  138.27       9.0   2025-05-11
6            Manubri          Sport     NaN       9.0   2025-06-16
7    Scarpe Sportive  Abbigliamento     NaN       2.0   2025-09-22
8           Cappello  Abbigliamento   46.17       7.0   2025-09-21
9        Caffettiera  Casa e Cucina   79.52       NaN   2025-01-02

################ DATAFRAME DOPO GESTIONE NAN ###################
            prodotto      categoria   prezzo  quantità data_vendita  \
0            Fumetto          Libri   31.610       8.0   2025-07-08   
1     Tappetin

C:\Users\raffy\AppData\Local\Temp\ipykernel_25060\378400873.py:88: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  oggi - df.groupby("prodotto")["data_vendita"].transform("min")
C:\Users\raffy\AppData\Local\Temp\ipykernel_25060\378400873.py:94: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('categoria')['prezzo'].transform('median')
C:\Users\raffy\AppData\Local\Temp\ipykernel_25060\378400873.py:145: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future def

Esercizio 3 – dataset completo mixato
•	Colonne: nome, email, data_iscrizione, età, stipendio, città, prodotto, categoria, vendite, giorni_attivi
•	Applica pipeline completa: pulizia stringhe, validazione, gestione valori mancanti, gestione outlier, feature engineering numerica e categoriale, estrazione dati temporali, calcolo differenze tra date
•	Usa apply, map e lambda per creare almeno due nuove colonne derivate combinando variabili diverse
•	Analizza memoria prima e dopo ottimizzazioni con tipi categorici e numerici

In [7]:
from datetime import datetime, timedelta 
import random 
import numpy as np 
import pandas as pd 

# impostazione del seed per rendere i dati riproducibili
np.random.seed(42)
random.seed(42)

n_righe = 100

# liste e dizionari per i dati 

nomi = [
    "Marco",
    "Giulia",
    "Luca",
    "Francesca",
    "Alessandro",
    "Sofia",
    "Andrea",
    "Chiara",
    "Matteo",
    "Elena",
]
cognomi = [
    "Rossi",
    "Bianchi",
    "Russo",
    "Ferrari",
    "Esposito",
    "Colombo",
    "Ricci",
    "Marino",
    "Bruno",
    "Conti",
]
citta_list = [
    "Milano",
    "Roma",
    "Torino",
    "Napoli",
    "Bologna",
    "Firenze",
    "Palermo",
    "Verona",
    "Genova",
    "Bari",
]
domini_email = ["gmail.com", "yahoo.it", "outlook.com", "hotmail.it"]

prodotti_per_categoria = {
    "Elettronica": [
        "Smartphone",
        "Laptop",
        "Cuffie Bluetooth",
        "Smartwatch",
        "Tablet",
    ],
    "Abbigliamento": [
        "Maglietta",
        "Giacca",
        "Pantaloni",
        "Scarpe Sportive",
        "Cappello",
    ],
    "Casa e Cucina": [
        "Caffettiera",
        "Frullatore",
        "Set Pentole",
        "Lampada da Tavolo",
        "Aspirapolvere",
    ],
    "Libri": [
        "Romanzo Thriller",
        "Manuale Python",
        "Libro di Cucina",
        "Saggio Storico",
        "Fumetto",
    ],
    "Sport": [
        "Pallone da Calcio",
        "Tappetino Yoga",
        "Manubri",
        "Bicicletta",
        "Borraccia Termica",
    ],
}

categorie = list(prodotti_per_categoria.keys())
oggi = datetime.now()
dati = []

# Generazione casuale dei record

for _ in range(n_righe):
    nome_completo = f"{random.choice(nomi)} {random.choice(cognomi)}"
    email = f"{nome_completo.lower().replace(' ','.')}@{random.choice(domini_email)}"

    # Generazione data di iscrizione (ultimi 3 anni)
    giorni_iscrizione = random.randint(10, 1095)
    data_iscrizione = (oggi - timedelta(days = giorni_iscrizione)).date()

    eta = random.randint(18, 65)
    stipendio = round(random.uniform(18000.0, 75000.0),2)
    citta = random.choice(citta_list)

    # Scelta categoria e prodotto coerente
    categ = random.choice(categorie)
    prod = random.choice(prodotti_per_categoria[categ])

    # Vendite (ricavo generato dall'utente) e giorni d'attività 
    vendite = round(random.uniform(20.0, 3500.0), 2)

    # I giorni attivi devono essere inferiori o uguali ai giorni trascorsi dall'iscrizione
    giorni_attivi = random.randint(1, giorni_iscrizione)

    dati.append({
        "nome": nome_completo,
        "email": email,
        "data_iscrizione": data_iscrizione.strftime("%Y-%m-%d"),
        "età": eta,
        "stipendio": stipendio,
        "città": citta,
        "prodotto": prod,
        "categoria": categ,
        "vendite": vendite,
        "giorni_attivi": giorni_attivi,
    })

# Creazione DataFrame 
df = pd.DataFrame(dati)

print("\n###################### DATAFRAME ORIGINALE ############################")
print(df)

###########################################################
# 1. PULIZIA STRINGHE E VALIDAZIONE EMAIL
###########################################################

# Pulizia testo: rimozione spazi vuoti e formattazione maiuscole/minuscole
df["nome"] = df["nome"].map(
    lambda x : x.strip().title() if pd.notnull(x) else x
)

df["email"] = df["email"].map(
    lambda x : x.strip().lower() if pd.notnull(x) else x
)

# Validazione sintattica dell'email via Regex (restituisce  True/False)
regex_email = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-0.-]+\.[a-zA-Z]{2,}$"
df["is_email_valida"] = df["email"].str.match(regex_email, na=False)

###########################################################
# 2. GESTIONE VALORI MANCANTI (IMPUTAZIONE)
###########################################################

# Imputazione numerica con Mediana
df["stipendio"] = df["stipendio"].fillna(df["stipendio"].median())
df["età"] = df["età"].fillna(df["età"].median())
df["vendite"] = df["vendite"].fillna(df["vendite"].median())

# Imputazione temporale e categoriale con la Moda (valore più frequente)
df["città"] = df["città"].fillna(df["città"].mode()[0])
df["categoria"] = df["categoria"].fillna(df["categoria"].mode()[0])

###########################################################
# 3. GESTIONE OUTLIER (METODO IQR)
###########################################################

colonne_outlier = ["stipendio","vendite"]

for col in colonne_outlier:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1 
    limite_inf = Q1 - 1.5 * IQR 
    limite_sup = Q3 + 1.5 * IQR 

    df[f"is_outlier_{col}"] = (df[col] < limite_inf) | (df[col] > limite_sup)


    # Capping: sostituisce i valori fuori soglia con i limiti inf/sup 
    # df[col] = np.clip(df[col], limite_inf, limite_sup)

###########################################################
# 4. ESTRAZIONE DATI TEMPORALI E DIFFERENZE
###########################################################

df["data_iscrizione"] = pd.to_datetime(df["data_iscrizione"])

# Estrazione componenti della data
df["anno_iscrizione"] = df["data_iscrizione"].dt.year 
df["mese_iscrizione"] = df["data_iscrizione"].dt.month
df["giorno_settimana"] = df["data_iscrizione"].dt.dayofweek # 0 lunedi - 6 domenica

# Differenza temporale rispetto ad oggi 
oggi = pd.Timestamp.now().normalize()
df["giorni_da_iscrizione"] = (oggi - df["data_iscrizione"]).dt.days

# Rapporto di attività (giorni attivi vs giorni di permanenza)
df["ratio_attivita"] = np.where(
    df["giorni_da_iscrizione"] > 0,
    df["giorni_attivi"] / df["giorni_da_iscrizione"],
    0,
)

###########################################################
# 5. FEATURE ENGINEERING NUMERICA
###########################################################

# Trasformazione logaritmica
df["log_vendite"] = np.log1p(df["vendite"])
df["log_stipendio"] = np.log1p(df["stipendio"])

###########################################################
# 6. FEATURE ENGINEERING CATEGORIALE (ENCODING)
###########################################################

# One - Hot encoding per la colonna categoria
df_categoria_encoded = pd.get_dummies(
    df["categoria"], prefix = "cat", dtype = int
)

df = pd.concat([df, df_categoria_encoded], axis=1)

# Frequency encoding per la colonna città
freq_citta = df["città"].value_counts(normalize=True)
df["citta_freq_encode"] = df["città"].map(freq_citta)

###########################################################
# CONTROLLO FINALE DEL DATASET TRASFORMATO
###########################################################

print("\n########### PANORAMICA PRIME RIGHE DATASET COMPLETO ###########")
print(df)

###########################################################
# 7. NUOVE COLONNE DERIVATE (apply, map, lambda)
###########################################################

# 1. Media vendite giornaliere (combina vendite e giorni_attivi tramite apply + lambda su righe)
df["media_vendite_giornaliere"] = df.apply(
    lambda row: round(row["vendite"]/row["giorni_attivi"], 2)
    if row["giorni_attivi"] > 0
    else 0,
    axis=1,
)

# 2. Tag geografico-prodotto (combina città e categoria tramite apply + lambda su righe)
df["tag_citta_categoria"] = df.apply(
    lambda row: f"{row['città'].upper()}_{row['categoria'].replace(' ','').upper()}",
    axis=1,
)

# 3. Rapporto vendite/stipendio formattato(combina vendite e stiepndio tramite) apply + lambda)
df["impatto_spesa_stipendio_%"] = df.apply(
    lambda row: round((row["vendite"]/row["stipendio"]) * 100, 2), axis=1
)

# 4. Normalizzazione fascia di età tramite map e lambda su colonna derivata
mappa_fasce = {True: "Senior", False:"junior"}
df["fascia_eta"] = (df["età"] >= 40).map(mappa_fasce)

print("\n########### DATASET CON NUOVE COLONNE DERIVATE ###########")
print(
    df[
        [
            "nome",
            "media_vendite_giornaliere",
            "tag_citta_categoria",
            "impatto_spesa_stipendio_%",
            "fascia_eta",
        ]
    ]
)


###########################################################
# MISURAZIONE DELLA MEMORIA
###########################################################

memoria_mb = df.memory_usage(deep =True).sum() / (1024**2)
print("PRIMA dell'ottimizzazione")
print(f"Totale: {memoria_mb:.2f} MB\n")

# Ottimizzazione dei tipi di dati

df_ottimizzato = df.copy()

# Ottimizzazione Categoriali (Stringhe/Oggetti a bassa cardinalità)
colonne_categoria = ["città","categoria","prodotto"]

for col in colonne_categoria:
    if col in df_ottimizzato.columns:
        df_ottimizzato[col] = df_ottimizzato[col].astype("category")


# Ottimizzazione tipi Numerici (Downcasting)
# Converte i float64 a float32 p float16 e gli int64 a int ridotti

for col in df_ottimizzato.select_dtypes(include=["int64","int32"]).columns:
    df_ottimizzato[col] = pd.to_numeric(df_ottimizzato[col], downcast="integer")

###########################################################
# 3. ANALISI COMPARATIVA
###########################################################

memoria_mb_dopo = df_ottimizzato.memory_usage(deep =True).sum() / (1024**2)
print("DOPO dell'ottimizzazione")
print(f"Totale: {memoria_mb_dopo:.2f} MB\n")

risparmio_percentuale = ((memoria_mb - memoria_mb_dopo) / memoria_mb) * 100
print(f"Risparmio netto di memoria: {risparmio_percentuale:.2f}%")


###################### DATAFRAME ORIGINALE ############################
                 nome                          email data_iscrizione  età  \
0        Giulia Rossi       giulia.rossi@outlook.com      2025-04-12   32   
1       Marco Bianchi         marco.bianchi@yahoo.it      2025-05-07   50   
2      Elena Esposito       elena.esposito@gmail.com      2025-10-04   62   
3      Giulia Bianchi      giulia.bianchi@hotmail.it      2026-02-09   40   
4        Giulia Ricci         giulia.ricci@gmail.com      2025-01-03   58   
..                ...                            ...             ...  ...   
95  Alessandro Marino  alessandro.marino@outlook.com      2026-08-07   47   
96     Elena Esposito      elena.esposito@hotmail.it      2025-04-01   47   
97      Elena Colombo      elena.colombo@outlook.com      2024-12-30   19   
98       Elena Marino       elena.marino@outlook.com      2025-05-12   56   
99       Luca Bianchi         luca.bianchi@gmail.com      2024-12-02   46   

  